<a href="https://colab.research.google.com/github/JoaquinMayorga/Movie-Recommendation-System/blob/main/Movie_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Movie Recommendation System</h1>

#<h2>Introduction</h3>

Have you ever watched a movie, loved it, and wanted recommendations on a similar movie to watch? This project aims to help you do that. Using the "*TMDB + IMDB Merged Movies Dataset*" created by Tejas Garg, we'll build a model using CountVectorizer that recommends similar movies based on a movie you give it.


The "TMDB + IMDB Merged Movies Dataset" created by Tejas Garg is under the [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) license.

Link to dataset:
https://www.kaggle.com/datasets/ggtejas/tmdb-imdb-merged-movies-dataset

#<h2>Part 1: Preparing the Data</h3>

# <h3>Loading the Data</h3>

In [1]:
import pandas as pd

When loading the dataset, I used **nrows** in the parameter to only load a certain amount of columns. I will explain why I did this in the next section.

In [2]:
# loading the datset
df = pd.read_csv('/content/TMDB  IMDB Movies Dataset.csv', nrows=10000)
df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2813160,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2523648,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3163534,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1496283,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1556844,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


# <h3>Cleaning the Data</h3>

The biggest challenges we will be facing in this project isn't training the recommendation system itself; The biggest challenge is using this dataset due to it's large size. This can lead to crashing and excess ram usage. Remember when we used **nrows** to load a certain amount of data? The reason we did this is to significantly lower memory usage and the time it takes to load the data. Without nrows, the memory usage was around 650 MB. That is a high amount of memory usage. Not only would it take a while to load the data, it would lead to crashing later on in project development. Since we used **nrows**, let's see how big this dataset is now.

In [3]:
df.info(verbose=False, memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 29 entries, id to cast
dtypes: bool(1), float64(3), int64(6), object(19)
memory usage: 18.5 MB


As we can see, memory usage is is significantly lower but we want to reduce it more.

Luckily for this project, we're only going to build a Vector model to recommend movies based on text. So we can first reduce the dataset by only including the string columns we need.

In [4]:
df = df[['title', 'overview', 'genres', 'keywords', 'directors']]
df.head()

,title,overview,genres,keywords,directors
0,Inception,"Cobb, a skilled thief who commits corporate es...","Action, Science Fiction, Adventure","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan
1,Interstellar,The adventures of a group of explorers who mak...,"Adventure, Drama, Science Fiction","rescue, future, spacecraft, race against time,...",Christopher Nolan
2,The Dark Knight,Batman raises the stakes in his war on crime. ...,"Drama, Action, Crime, Thriller","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan
3,Avatar,"In the 22nd century, a paraplegic Marine is di...","Action, Adventure, Fantasy, Science Fiction","future, society, culture clash, space travel, ...",James Cameron
4,The Avengers,When an unexpected enemy emerges and threatens...,"Science Fiction, Action, Adventure","new york city, superhero, shield, based on com...",Joss Whedon


In [5]:
df.info(verbose=False, memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 5 entries, title to directors
dtypes: object(5)
memory usage: 7.5 MB


By removing unnecessary columns, we were able to further reduce the memory usage. Let's see if we can reduce it further.

Let's check to see if there are any missing values.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   title      10000 non-null  object
 1   overview   9999 non-null   object
 2   genres     9997 non-null   object
 3   keywords   9551 non-null   object
 4   directors  9999 non-null   object
dtypes: object(5)
memory usage: 390.8+ KB


In [7]:
df.isna().sum()

,0
title,0
overview,1
genres,3
keywords,449
directors,1


This dataset does contain missing values. Let's remove them.

In [8]:
# Removing null values
df.dropna(inplace=True)

In [9]:
df.isna().sum()

,0
title,0
overview,0
genres,0
keywords,0
directors,0


Now let's see if there are any duplicate values.

In [10]:
df.duplicated().sum()

np.int64(8)

The output shows us there are duplicate values. So let's remove them and confirm if any duplicate values remain.

In [11]:
df = df.drop_duplicates()

In [12]:
df.duplicated().sum()

np.int64(0)

Let's check the memory usage one last time to see how much it's been reduced to.

In [13]:
df.info(verbose=False, memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 9541 entries, 0 to 9998
Columns: 5 entries, title to directors
dtypes: object(5)
memory usage: 7.3 MB


Adding a new column to the DataFrame called **features** that combines all of the words in the other columns.

In [14]:
df['features'] = df['title'] + ' ' + df['overview'] + ' ' + df['genres'] + ' ' + df['keywords'] + ' ' + df['directors']

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9541 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   title      9541 non-null   object
 1   overview   9541 non-null   object
 2   genres     9541 non-null   object
 3   keywords   9541 non-null   object
 4   directors  9541 non-null   object
 5   features   9541 non-null   object
dtypes: object(6)
memory usage: 521.8+ KB


Tokenizing and vectorizing the **features** column.

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

In [17]:
vectorizer = CountVectorizer(stop_words='english', min_df=20)
word_matrix = vectorizer.fit_transform(df['features'])
word_matrix.shape

(9541, 3387)

In [18]:
# Making the similarity matrix that will be used in the next section
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(word_matrix)

# <h2>Part 2: Building the Recommendation Model</h2>

# <h3> Building the Model</h3>

Now that we cleaned the data and reduced memory usage for more optimal performance, we can build the model to generate movie recommendations.

In [19]:
# Defining the function to get movie recommendations
def get_recommendations(title, df, sim, count=10):

  # Get the row index of the specified title in the DataFrame
  index = df.index[df['title'].str.lower() == title.lower()]

  # Returns an empty list if there is no entry for the specified title
  if (len(index) == 0):
    return[]

  # Gathers the corresponding row in the similarity matrix
  similarities = list(enumerate(sim[index[0]]))

  # Sorts the similarities in descending order
  recommendations = sorted(similarities, key=lambda x: x[1], reverse=True)

  # Get the top n recommendations, ignoring the first entry in the list since
  # it corresponds to the title itself (and thus has a similarity of 1.0)
  top_recs = recommendations[1:count + 1]

  # Generates a list of titles from the indexes in top_recs
  titles = []

  for i in range(len(top_recs)):
    title = df.iloc[top_recs[i][0]]['title']
    titles.append(title)

  # Return the list of titles
  return titles

# <h3>Movie Recommendations</h3>

Now that we've built the model, we can finally test it to get movie recommendations based on a movie we give it.

In [20]:
get_recommendations('The Dark Knight', df, sim)

['Batman',
 'Batman: Mask of the Phantasm',
 'The Dark Knight Rises',
 'Batman Begins',
 'Batman Forever',
 'The Batman',
 'Batman: Under the Red Hood',
 'Batman: The Long Halloween, Part One',
 'Batman: The Long Halloween, Part Two',
 'Batman Returns']

In [21]:
get_recommendations('Avatar', df, sim)

['Cosmic Sin',
 'Space Chimps',
 'Alien',
 'Lost in Space',
 'Soldier',
 'Aliens',
 'Dune',
 'Lifeforce',
 'Mission to Mars',
 'Lightyear']

In [22]:
get_recommendations('Interstellar', df, sim)

['Lost in Space',
 'Frequently Asked Questions About Time Travel',
 'Space Chimps',
 'Stowaway',
 'A Walk in Time',
 'Salyut-7',
 'Return to Space',
 'Planet of the Apes',
 'Space Sweepers',
 'See You Yesterday']

In [23]:
get_recommendations('Dumb and Dumber', df, sim)

['Dumb and Dumber To',
 'Friends with Money',
 'Mad Money',
 'Bad Trip',
 'The Road Within',
 '3 Men and a Little Lady',
 'Tommy Boy',
 'Road Trip',
 'The Brass Teapot',
 'Red Rock West']